<a href="https://colab.research.google.com/github/juanitapinedaguti/Integracion-de-Datos-y-Prospectiva/blob/main/Caso_de_estudio_US_Health_Insurance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Caso de estudio-US Health Insurance**

#### Una aseguradora de salud con cobertura nacional en los Estados Unidos quiere mejorar la cobertura preventiva de sus afiliados. La entidad necesita identificar patrones de riesgo epidemiológico para reorganizar la oferta de servicios médicos e incentivos de salud, en lugar de trasladar a los pacientes, para lo cual quiere crear unos clusters en salud en donde quiere analizar la distribución territorial de estos grupos por regiones para optimizar la asignación de recursos y programas preventivos.

#### **Descripción de las variables**
#### **Age:** Edad del beneficiario principal
#### **Sex:** Género del titular del seguro, femenino / masculino
#### **BMI:** Índice de masa corporal (IMC), que permite comprender si el peso corporal es relativamente alto o bajo
#### **Children:** Número de hijos cubiertos por el seguro de salud / Número de dependientes
#### **Smoker:** Fumador / No fumador
#### **Region:** Área de residencia del beneficiario en Estados Unidos: noreste, sureste, suroeste o noroeste
#### **Charges:** Costos médicos individuales facturados por el seguro de salud



####**Abstracción**
####✓ La entidad quiere identificar el número de clusters que pueden ser eficientes para la reorganización de la oferta de servicios médicos.
####✓ Quiere caracterizar las variables clínicas y los perfiles de los pacientes de acuerdo con sus variables epidemiológicas (Variables de Entrada) y de costo siniestro (Variable de Salida).
####✓ La entidad quiere conocer la distribución porcentual por región y por siniestralidad (Variables Salida).
####✓ Para la entrega del reto, es importante organizar el repositorio en GitHub con los retos desarrollados.

####**Técnica a utilizar**
####K-Medoids (o el algoritmo PAM, Partitioning Around Medoids) es un método de aprendizaje no supervisado que agrupa n objetos en 𝑘 clusters seleccionando puntos reales del conjunto de datos (medoides) como centros de cada grupo.

####**Archivo**: US Health Insurance Dataset

# 0. Cargar la librerías de trabajo

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1. Carga de base de datos

In [2]:
nxl="/content/drive/MyDrive/INTEGRACION Y PROSPECTIVA/insurance.xlsx"
XDB=pd.read_excel(nxl,sheet_name=0)
XDB.dropna()
XDB.head()

XDB2=XDB.iloc[:,[0,1,2,3,4,5,6]].copy() # Acá están TODAS las variables
XDB2.head()

XDB=XDB.iloc[:,[0,1,2,3,4,6]].copy() # Acá solo están las variables de entrada
XDB.head()

,age,sex,bmi,children,smoker,charges
0,18,male,23.21,0,no,1121.8739
1,18,male,30.14,0,no,1131.5066
2,18,male,33.33,0,no,1135.9407
3,18,male,33.66,0,no,1136.3994
4,18,male,34.10,0,no,1137.0110


#1.1 Pre-procesamiento de los datos

#####Codificar las variables categóricas

In [3]:
# Convertir variables categóricas a numéricas

XDB['sex'] = XDB['sex'].map({'female': 1, 'male': 0})
XDB2['sex'] = XDB2['sex'].map({'female': 1, 'male': 0})
XDB['smoker'] = XDB['smoker'].map({'no': 0, 'yes': 1})
XDB2['smoker'] = XDB2['smoker'].map({'no': 0, 'yes': 1})

XDB.head()
XDB.shape

(1338, 6)

# 2. Creación de los clusters de salud por región

In [4]:
#Identificar las regiones
Xreg = XDB2['region'].unique()
Xreg

array(['southeast', 'southwest', 'northwest', 'northeast'], dtype=object)

In [5]:
# Creación de clusters
XCm = np.zeros((len(Xreg),6)) # No se incluye la variable region
XDB_features = XDB.copy()

# Fase 0 de integración: reconocer los valores de entrada por región
for i, region_name in enumerate(Xreg):
    print(i, region_name)

    filas = np.where(XDB2['region'] == region_name)[0]

    XCm[i,:] = np.mean(XDB.iloc[filas,:], axis=0)

XCma = XCm.copy()
display(XCma)

0 southeast
1 southwest
2 northwest
3 northeast


array([[3.89395604e+01, 4.80769231e-01, 3.33559890e+01, 1.04945055e+00,
        2.50000000e-01, 1.47354114e+04],
       [3.94553846e+01, 4.98461538e-01, 3.05966154e+01, 1.14153846e+00,
        1.78461538e-01, 1.23469374e+04],
       [3.91969231e+01, 5.04615385e-01, 2.91997846e+01, 1.14769231e+00,
        1.78461538e-01, 1.24175754e+04],
       [3.92685185e+01, 4.96913580e-01, 2.91735031e+01, 1.04629630e+00,
        2.06790123e-01, 1.34063845e+04]])

In [6]:
XCma_df = pd.DataFrame(
    XCma,
    index=Xreg,
    columns=XDB.columns
)

display(XCma_df)

,age,sex,bmi,children,smoker,charges
southeast,38.939560,0.480769,33.355989,1.049451,0.250000,14735.411438
southwest,39.455385,0.498462,30.596615,1.141538,0.178462,12346.937377
northwest,39.196923,0.504615,29.199785,1.147692,0.178462,12417.575374
northeast,39.268519,0.496914,29.173503,1.046296,0.206790,13406.384516


#### Al comparar los perfiles promedio de las cuatro regiones, vemos que la edad, el genero y el número de hijos presentan valores  similares. Las principales diferencias se encuentran en el BMI, la proporción de fumadores y los gastos médicos. La región Southeast presenta los mayores charges promedio ($14,735.41), junto con el BMI promedio más alto (33.36) y la mayor proporción de fumadores (25%). Por otra parte, Northeast tiene el segundo mayor gasto y la segunda mayor proporción de fumadores. Estos resultados sugieren que hay una posible relación entre variables como el tabaquismo y el BMI con los costos médicos. Sin embargo, se requiere un análisis adicional para determinar su influencia.

# 3. Creacion de 3 clusters (segmentos) para ver los perfiles de las personas que tienen charges altos, medios y bajos.

In [7]:
from sklearn.cluster import KMeans

# Crear los 3 clusters usando charges
X_charges = XDB2[['charges']].copy()

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
XDB2['Cluster_num'] = kmeans.fit_predict(X_charges)

# Calcular el promedio de charges de cada cluster
charges_cluster = XDB2.groupby('Cluster_num')['charges'].mean().sort_values()

# Nombrar los clusters según su nivel de charges
orden = charges_cluster.index.tolist()

nombres_cluster = {
    orden[0]: 'Bajo',
    orden[1]: 'Medio',
    orden[2]: 'Alto'
}

XDB2['Cluster'] = XDB2['Cluster_num'].map(nombres_cluster)

# Ver cómo quedaron los clusters
resumen_clusters = XDB2.groupby('Cluster')['charges'].agg(
    ['min', 'max', 'mean', 'count']
)

resumen_clusters = resumen_clusters.reindex(['Bajo', 'Medio', 'Alto'])

display(resumen_clusters.round(2))

,min,max,mean,count
Cluster,,,,
Bajo,1121.87,12404.88,6303.04,870
Medio,12430.95,29523.17,18525.65,306
Alto,30063.58,63770.43,40761.31,162


In [8]:
perfil_cluster = XDB2.groupby('Cluster', observed=False).agg(
    BMI_promedio=('bmi', 'mean'),
    Proporción_fumadores=('smoker', lambda x: x.mean() * 100)
)

perfil_cluster = perfil_cluster.reindex(['Bajo', 'Medio', 'Alto'])

display(perfil_cluster.round(3))

,BMI_promedio,Proporción_fumadores
Cluster,,
Bajo,30.438,0.000
Medio,29.091,39.869
Alto,34.845,93.827


####Al cruzar los niveles de costos médicos con las características de los asegurados, se observa una fuerte diferencia en la proporción de fumadores entre los tres clusters. El cluster de costos bajos presenta 0% de fumadores, mientras que esta proporción aumenta a 39.87% en el cluster medio y a 93.83% en el cluster alto. El BMI, por su parte no presenta un incremento progresivo entre los tres grupos, aunque alcanza su mayor valor en el cluster de costos altos (34.85). Esto sugiere que el tabaquismo es una de las características que más distingue a los perfiles asociados con mayores costos médicos, mientras que un BMI elevado podría adquirir mayor relevancia cuando se combina con otras características, como ser fumador.

In [10]:
# Distribución de los clusters de charges por región

region_cluster = pd.crosstab(
    XDB2['region'],
    XDB2['Cluster'],
    normalize='index'
) * 100

# Ordenar las columnas
region_cluster = region_cluster[['Bajo', 'Medio', 'Alto']]

display(region_cluster.round(2))

Cluster,Bajo,Medio,Alto
region,,,
northeast,61.11,28.09,10.80
northwest,67.38,23.69,8.92
southeast,61.26,21.98,16.76
southwest,70.77,17.85,11.38


####Southeast es la región que más llama la atención. Tiene el mayor porcentaje de personas en el cluster Alto: 16.76%. Esto es bastante superior a Northeast (10.80%), Southwest (11.38%) y especialmente Northwest (8.92%). Esto coincide con lo que vimos antes: Southeast también tenía el charges promedio regional más alto. Por eso, en esta región tendría más sentido que la entidad se enfoque en programas dirigidos a tabaquismo y manejo de factores asociados con BMI elevado, porque es donde proporcionalmente encontramos más personas dentro del cluster de charges altos.

# 4. Tabla consolidada

In [11]:
# Crear una tabla resumen por cluster

tabla_final = XDB2.groupby('Cluster', observed=False).agg(
    Charge_minimo=('charges', 'min'),
    Charge_maximo=('charges', 'max'),
    Charge_promedio=('charges', 'mean'),
    Cantidad_personas=('charges', 'count'),
    BMI_promedio=('bmi', 'mean'),
    Proporcion_fumadores=('smoker', lambda x: x.mean() * 100)
)

# Calcular la distribución de regiones DENTRO de cada cluster
regiones = pd.crosstab(
    XDB2['Cluster'],
    XDB2['region'],
    normalize='index'
) * 100

# Cambiar nombres de las columnas
regiones.columns = [
    'Region_' + str(col) + '_%'
    for col in regiones.columns
]

# Unir las dos tablas
tabla_final = tabla_final.join(regiones)

# Ordenar los clusters
tabla_final = tabla_final.reindex(['Bajo', 'Medio', 'Alto'])

# Mostrar resultado
display(tabla_final.round(2))

,Charge_minimo,Charge_maximo,Charge_promedio,Cantidad_personas,BMI_promedio,Proporcion_fumadores,Region_northeast_%,Region_northwest_%,Region_southeast_%,Region_southwest_%
Cluster,,,,,,,,,,
Bajo,1121.87,12404.88,6303.04,870,30.44,0.00,22.76,25.17,25.63,26.44
Medio,12430.95,29523.17,18525.65,306,29.09,39.87,29.74,25.16,26.14,18.95
Alto,30063.58,63770.43,40761.31,162,34.85,93.83,21.60,17.90,37.65,22.84


# 5. Estrategias sugeridas para cada cluster

####**Cluster 1 (costos bajos):**
####**Plan Mantener**
#### La aseguradora podría automatizar gran parte de la prevención: mensajes personalizados, recordatorios anuales de controles, contenido sobre alimentación y ejercicio, incentivos por completar actividades preventivas y monitoreo periódico para detectar personas cuyo perfil empiece a cambiar. De esta forma se puede conservar el perfil de bajo costo y detectar tempranamente señales de riesgo.

####**Cluster 2 (costos medios):**
####**Plan Prevenir**
Este es el segmento en el que hay mas oportunidades de prevencióncon dado que tienen costos  mayores y casi 4 de cada 10 son fumadores, pero todavía están lejos del costo promedio del grupo de charges altos.

Para este grupo se podría implementar una estrategia de prevención temprana y de monitoreo periódico de factores de riesgo con el fin de reducir el riesgo de que personas de costo medio evolucionen hacia perfiles de costo alto.

Primero, ingresar a los fumadores a un programa enfocado en la reducción del tabaquismo que consista en identificar a los fumadores del segmento y ofrecerles acompañamiento para dejar de fumar, con seguimiento de su progreso. Por otro lado, las personas con BMI elevado podrían recibir orientación nutricional y apoyo en actividad física. Además, un seguimiento semestral permitiría identificar aumentos en costos o cambios en el perfil de los asegurados y asi, poder mitigar los riegsos antes de que alcance un perfil de alto costo.

####**Cluster 3:**
####**Plan priorizar**
Con el objetivo de gestionar activamente los perfiles de mayor costo e intervenir sobre factores de riesgo asociados con ellos. Este grupo representa solo 162 de 1,338 personas (~12%), pero tiene charges promedio superiores a 40,000. Aquí sí tiene sentido gastar más recursos por persona. Al ser un grupo mas pequeño, se sugiere asignar gestores de casos para hacer seguimiento individual, coordinar controles médicos y conectar a los asegurados con programas específicos. Dado que 93.83% son fumadores y su BMI promedio es 34.85, los programas de cesación del tabaquismo y manejo de peso tambien seran claves.

#6. Estrategias por región
####Asignar diferentes niveles de recursos según el perfil de costos y su concentración geográfica:

#### **Southeast:** implementar el plan priorizar, ya que es la región con mayor proporción de personas en el segmento Alto (16.76%).

#### **Northeast:** fortalecer el plan prevenir, porque presenta la mayor proporción de personas en el segmento Medio (28.09%), lo que representa una oportunidad para intervenir antes de que sus costos aumenten.

#### **Northwest:** enfocarse principalmente en el plan mantener, ya que presenta la menor proporción de personas en el segmento Alto (8.92%).

#### **Southwest:** concentrar los esfuerzos en el plan mantener, utilizando estrategias preventivas de bajo costo y amplio alcance, ya que 70.77% de las personas se encuentran en el segmento Bajo.
